In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys # to give information
from selenium.webdriver.common.by import By
import re
import pandas as pd
import numpy as np

In [2]:
option = Options()
option.add_argument('--start-maximized')

driver = webdriver.Chrome(options=option)

In [3]:
driver.get('https://www.pepperfry.com/site_product/search?q=recliners')

In [4]:
def get_recliner_details(driver):
    seater = []
    color = []
    model = []
    brand_names = []
    ratings = []
    warrantys = []
    cost = []
    org_cost = []
    discounts = []
    EMIs = []
    shipping_days = []
    
    recliner_heading = driver.find_elements(By.CSS_SELECTOR, '.product-name color-tertiary text-md font-medium ng-star-inserted'.replace(' ', '.'))
    for h in recliner_heading:
        rec_detail = h.text
        # print(rec_detail)
        
        m = re.findall(r'Rx\d\s*(\w+)|([a-zA-Z]+)', rec_detail)
        if m:
            m = m[0][0] if m[0][0] else m[0][1]
        else:
            m = None
        model.append(m)
        
        seat = re.findall(r'(\d{1})\s*[sS]eater|(\d{1})S', rec_detail)
        if seat:
            seat = seat[0][0] if seat[0][0] else seat[0][1]
        else:
            seat = None
        seater.append(seat)

        cl = re.findall(r'(\w+)\s*Colou?r|(\w+)\s*Finish', rec_detail)
        if cl:
            cl = cl[0][0] if cl[0][0] else cl[0][1]
        else:
            cl = None
        color.append(cl)

    brand = driver.find_elements(By.CSS_SELECTOR, '.product-brand text-xs color-secondary font-medium ng-star-inserted'.replace(' ', '.'))
    for br in brand:
        b_name = re.findall(r'By\s*(.+)', br.text)[0]
        brand_names.append(b_name)

    rating = driver.find_elements(By.CSS_SELECTOR, '.text-sm'.replace(' ', '.'))
    for r in rating:
        if r.text:
            rat = re.findall(r'^(\d{1})$|^(\d{1}\.\d{1})$', r.text)
            if rat:
                rat = rat[0][0] if rat[0][0] else rat[0][1]
            else:
                rat = None
            ratings.append(rat)
    warranty = driver.find_elements(By.CSS_SELECTOR, '.color-secondary text-xs font-normal ng-star-inserted'.replace(' ', '.'))
    for w in warranty:
        wr = re.findall(r'(\d{2})\-Month Warranty Available', w.text)
        wr = wr[0] if wr else None
        warrantys.append(wr)

    cost_ = driver.find_elements(By.CSS_SELECTOR, '.product-offer-price font-bold text-xl ng-star-inserted'.replace(' ', '.'))
    for c in cost_:
        ct = re.findall(r'\d', c.text)
        ct = int(''.join(ct))
        cost.append(ct)

    org_cost_ = driver.find_elements(By.CSS_SELECTOR, '.product-mrp-price line-through text-lg color-secondary ng-star-inserted'.replace(' ', '.'))
    for c in org_cost_:
        ct = re.findall(r'\d', c.text)
        ct = int(''.join(ct))
        org_cost.append(ct)

    discount = driver.find_elements(By.CSS_SELECTOR, '.product-discount color-green text-lg font-bold ng-star-inserted'.replace(' ', '.'))
    for d in discount:
        dis = re.findall(r'(\d{1,2})\%\s*off', d.text)[0]
        discounts.append(dis)

    EMI = driver.find_elements(By.CSS_SELECTOR, '.font-normal'.replace(' ', '.'))
    for em in EMI:
        if em.text.startswith('EMI'):
            emi = re.findall(r'EMI starting from ₹(.+)\/month', em.text)[0]
            emi = int(''.join(re.findall(r'\d', emi)))
            EMIs.append(emi)

    shipping = driver.find_elements(By.CSS_SELECTOR, '.font-bold'.replace(' ', '.'))
    for sh in shipping:
        if 'shipping' in sh.text.lower() or 'ships' in sh.text.lower():
            sh_dys = re.findall(r'Express\s*Shipping\s*in\s*(\d{1,2}) days|Ships\s*in\s*(\d{1,2})\s*days', sh.text)
            if sh_dys:
                sh_dys = sh_dys[0][0] if sh_dys[0][0] else sh_dys[0][1]
            else:
                sh_dys = None
            shipping_days.append(sh_dys)
    
    return seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days

In [5]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 45 36 40 40 40 40 26


In [8]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Sleepyhead,Manaual,Brown,1,NaN,12,14249,19999,29,685,2
1,Casacraft from Pepperfry,Norton,Brown,1,NaN,36,40999,50999,20,1969,3
2,Royaloak,Nelson,NaN,1,NaN,12,13990,30000,53,672,2
3,Sleepyhead,Manaual,Black,1,NaN,36,14249,19999,29,685,2
4,Duroflex,Avalon,Green,1,NaN,12,18049,25399,29,867,NaN
5,Nilkamal,Sierra,Brown,1,NaN,36,17490,47900,63,840,NaN
6,Casacraft from Pepperfry,Norton,Beige,1,NaN,36,31999,42999,26,1537,4
7,Green Soul,Flexy,Grey,1,NaN,36,15990,27990,43,768,2
8,Casacraft from Pepperfry,Norton,Brown,3,NaN,36,79999,99999,20,3841,2
9,Duroflex,Avalon,Orange,1,NaN,36,18619,26199,29,894,4


In [9]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [10]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 46 38 40 40 40 40 22


In [11]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df2 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df2

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Duroflex,Avalon,Green,1,NaN,36,23464,32999,29,1127,NaN
1,Casacraft from Pepperfry,Norton,Grey,3,NaN,36,79999,99999,20,3841,NaN
2,Mintwud from Pepperfry,Wakizashi,Blue,3,NaN,36,65999,84999,22,3169,NaN
3,Nilkamal,Matt,Blue,1,NaN,12,17990,74900,76,864,3
4,Casacraft from Pepperfry,Norton,Grey,1,NaN,36,40999,51999,21,1969,2
5,Royaloak,Denver,Beige,2,NaN,12,72500,155000,53,3481,NaN
6,Royaloak,Wave,Beige,1,NaN,12,23000,45000,49,1105,3
7,Casacraft from Pepperfry,Norton,Brown,1,NaN,36,31999,42999,26,1537,2
8,Casacraft from Pepperfry,Norton,Brown,2,NaN,36,79999,99999,20,3841,NaN
9,Royaloak,AUSTIN,NaN,1,NaN,12,29000,62000,53,1393,2


In [12]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [13]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 35 38 40 40 40 40 23


In [14]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df3 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df3

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Mintwud from Pepperfry,Kodachi,Beige,1,NaN,36,24999,33280,25,1201,NaN
1,Mintwud from Pepperfry,Taki,Cream,1,NaN,36,26879,38999,31,1291,NaN
2,Casacraft from Pepperfry,Norton,Brown,3,NaN,36,69999,84999,18,3361,NaN
3,Casacraft from Pepperfry,Norton,Grey,3,NaN,36,69999,84999,18,3361,NaN
4,Casacraft from Pepperfry,Norton,Beige,3,NaN,36,69999,84999,18,3361,NaN
5,Mintwud from Pepperfry,Yamamoto,Beige,1,NaN,12,24999,37899,34,1201,NaN
6,Durian,Valerano,Brown,1,NaN,60,76320,127200,40,3665,NaN
7,Casacraft from Pepperfry,Santos,Grey,3,NaN,36,74999,102999,27,3601,NaN
8,Casacraft from Pepperfry,Tierra,Tan,1,NaN,12,42999,58999,27,2065,NaN
9,Casacraft from Pepperfry,Carerra,Brown,1,NaN,36,36999,45999,20,1777,NaN


In [15]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [16]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 37 38 40 40 40 40 22


In [17]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df4 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df4

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Casacraft from Pepperfry,Norton,Brown,3,NaN,36,89999,106999,16,4322,NaN
1,Casacraft from Pepperfry,Norton,Brown,2,NaN,36,104999,129999,19,5042,NaN
2,Casacraft from Pepperfry,Norton,Brown,2,NaN,36,69999,79999,13,3361,NaN
3,Casacraft from Pepperfry,Norton,Beige,2,NaN,36,79999,99999,20,3841,NaN
4,Casacraft from Pepperfry,Norton,Brown,1,NaN,36,45999,58999,22,2209,3
5,Royaloak,Dallas,Brown,3,NaN,12,97000,200000,52,4658,3
6,Royaloak,Dallas,Brown,2,NaN,12,63000,135000,53,3025,3
7,Royaloak,Dallas,Brown,1,NaN,12,40000,87000,54,1921,NaN
8,Durian,Splendor,Green,3,NaN,60,268920,448200,40,12912,3
9,Durian,Splendor,Green,2,NaN,60,239280,398800,40,11489,3


In [18]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [19]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 43 34 40 40 40 40 30


In [20]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df5 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df5

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Casacraft from Pepperfry,Blacksburg,Beige,2,NaN,36,199999,269999,26,9603,2
1,Casacraft from Pepperfry,Riyadh,Green,2,NaN,36,209999,354999,41,10083,NaN
2,Duroflex,Avalon,Grey,1,NaN,12,31999,44000,27,1537,2
3,Recliners India,Omega,Grey,2,NaN,12,73600,217350,66,3534,2
4,Recliners India,Omega,Grey,1,NaN,12,42550,126500,66,2043,2
5,Lezino,Faric,Black,1,NaN,12,56000,85000,34,2689,2
6,Mintwud from Pepperfry,Wakizashi,Grey,2,NaN,36,46999,53999,13,2257,2
7,Durian,Vivian,Green,1,NaN,60,129600,216000,40,6223,2
8,LA-Z-BOY,Dreamtime,Chestnut,1,NaN,20,180500,190000,5,8667,2
9,LA-Z-BOY,Dreamtime,Charcoal,1,NaN,20,180500,190000,5,8667,2


In [21]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [22]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 48 37 40 40 40 40 20


In [23]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df6 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df6

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Royaloak,Regal,Brown,3,NaN,60,72000,170000,58,3457,3
1,Royaloak,Nexus,Brown,2,NaN,60,58000,135000,57,2785,3
2,Royaloak,Naples,Brown,2,NaN,12,78000,133000,41,3746,3
3,Durian,Valerano,Brown,3,NaN,12,160440,267400,40,7704,4
4,Durian,Valerano,Brown,2,NaN,24,129600,216000,40,6223,3
5,Nilkamal,Skelton,Brown,2,NaN,24,28900,63000,54,1388,2
6,Royaloak,Georgia,Coffee,3,NaN,24,78000,175000,55,3746,3
7,The Sleep Company,Luxe,Beige,1,NaN,12,30999,47690,35,1489,2
8,The Sleep Company,Luxe,Beige,1,NaN,12,35999,55390,35,1729,2
9,The Sleep Company,Luxe,Grey,1,NaN,36,30999,47690,35,1489,2


In [24]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [25]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 36 36 40 40 40 40 7


In [26]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df7 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df7

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Star India,Square,Brown,3,None,12,98999,128699,23,4754,NaN
1,Star India,Square,Brown,3,None,12,71999,86999,17,3457,3
2,Star India,Neo,Black,1,None,12,53999,67599,20,2593,3
3,Royaloak,Milano,NaN,3,None,12,150000,260000,42,7202,2
4,Royaloak,Milano,NaN,2,None,12,125000,215000,42,6002,2
5,Star India,Felso,Olive,NaN,None,60,130999,192171,32,6290,NaN
6,Interio By Godrej,Luster,Granite,NaN,None,36,207790,235700,12,9977,NaN


In [27]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [28]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

40 40 40 40 52 40 40 40 40 40 13


In [29]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df8 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df8

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Casacraft from Pepperfry,Atlantis,Grey,NaN,NaN,12,84999,114999,26,4082,None
1,Casacraft from Pepperfry,Montez,Beige,3,NaN,36,58999,79999,26,2833,None
2,Casacraft from Pepperfry,Montez,Beige,3,NaN,36,58999,79999,26,2833,None
3,Casacraft from Pepperfry,Montez,Green,NaN,NaN,36,79999,109999,27,3841,None
4,Casacraft from Pepperfry,Montez,Beige,NaN,NaN,36,79999,109999,27,3841,None
5,Casacraft from Pepperfry,Montez,Beige,NaN,NaN,36,79999,109999,27,3841,None
6,Casacraft from Pepperfry,Montez,Beige,NaN,NaN,36,79999,109999,27,3841,None
7,Casacraft from Pepperfry,Montez,Green,NaN,NaN,36,79999,109999,27,3841,None
8,Casacraft from Pepperfry,Montez,Beige,NaN,NaN,36,79999,109999,27,3841,None
9,Casacraft from Pepperfry,Montez,Beige,NaN,NaN,36,79999,109999,27,3841,None


In [30]:
driver.execute_script('window.scrollBy(0, 6500)')

next_page_button = driver.find_element(
    By.XPATH,
    '/html/body/app-root/main/app-category/pf-clip/div/div[2]/pf-clip-product-listing/div[3]/div/div/div[2]'
)
driver.execute_script("arguments[0].click();", next_page_button)

In [31]:
seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days = get_recliner_details(driver)
# print(seater, color, model, brand_names, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
print(len(seater), len(color), len(model), len(brand_names), len(ratings), len(warrantys), len(cost), len(org_cost), len(discounts), len(EMIs), len(shipping_days))

25 25 25 25 37 25 25 25 25 25 4


In [32]:
zipped = zip(brand_names, model, color, seater, ratings, warrantys, cost, org_cost, discounts, EMIs, shipping_days)
df9 = pd.DataFrame(zipped, columns = ['Brand', 'Model', 'Color', 'Seater', 'Rating', 'Warranty(Months)', 'Cost after Discount(%)', 'Original Cost', 'Discount(%)', 'EMI', 'Shipping Days'])
df9

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Woodsworth from Pepperfry,Serena,Blue,None,None,36,55999,72999,23,2689,None
1,Woodsworth from Pepperfry,Serena,Blue,None,None,36,64999,84999,24,3121,None
2,Woodsworth from Pepperfry,Serena,Blue,None,None,36,64999,84999,24,3121,None
3,Woodsworth from Pepperfry,Serena,Brown,None,None,36,55999,72999,23,2689,None


In [37]:
final_df = pd.concat((df, df2, df3, df4, df5, df6, df7, df8, df9)).reset_index().drop(columns=['index'])
final_df

,Brand,Model,Color,Seater,Rating,Warranty(Months),Cost after Discount(%),Original Cost,Discount(%),EMI,Shipping Days
0,Sleepyhead,Manaual,Brown,1,NaN,12,14249,19999,29,685,2
1,Casacraft from Pepperfry,Norton,Brown,1,NaN,36,40999,50999,20,1969,3
2,Royaloak,Nelson,NaN,1,NaN,12,13990,30000,53,672,2
3,Sleepyhead,Manaual,Black,1,NaN,36,14249,19999,29,685,2
4,Duroflex,Avalon,Green,1,NaN,12,18049,25399,29,867,NaN
...,...,...,...,...,...,...,...,...,...,...,...
162,Casacraft from Pepperfry,Larenzo,Pink,3,5,36,59999,79999,25,2881,None
163,Woodsworth from Pepperfry,Serena,Blue,None,None,36,55999,72999,23,2689,None
164,Woodsworth from Pepperfry,Serena,Blue,None,None,36,64999,84999,24,3121,None
165,Woodsworth from Pepperfry,Serena,Blue,None,None,36,64999,84999,24,3121,None


In [38]:
final_df.to_csv('Recliners.csv')